## **importation de data et libreries**

In [51]:
import pandas as pd 
import joblib
from sklearn.model_selection import train_test_split,GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from xgboost import XGBRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

df= pd.read_csv('../data/cleaned_cars.csv')
df = df.drop('name',axis=1)

## **separation des X et Y**

In [52]:
x= df.drop('selling_price',axis=1)
y= df['selling_price']


## **80% training 20% test**

In [53]:
x_train , x_test, y_train ,y_test = train_test_split(
    x,
    y,
    test_size=0.2,
    random_state=10
)

## **Preprocessing**

In [54]:
numeric_columns = [
    "year",
    "km_driven"
]

categorical_columns = [
    "fuel",
    "seller_type",
    "transmission",
    "owner"
]

preprocessor = ColumnTransformer(
    transformers=[

        (
            "numeric",
            StandardScaler(),
            numeric_columns
        ),
        (
            "categorical",
            OneHotEncoder(
                drop="first",
                handle_unknown="ignore"
            ),
            categorical_columns
        )
    ]
)


## **parametre Models**

In [55]:
models = {

    "Linear Regression": {
        "model": LinearRegression(),
        "params": {
            "model__fit_intercept": [True, False]
        }
    },
    "Random Forest": {
        "model": RandomForestRegressor(
            random_state=10
        ),
        "params": {
            "model__n_estimators": [100, 200],
            "model__max_depth": [None, 10, 20]
        }
    },


    "SVR": {

        "model": SVR(),

        "params": {
            "model__C": [10, 100, 1000],
            "model__epsilon": [0.01, 0.1],
            "model__gamma": ["scale", "auto"]
        }
    },


    "XGBoost": {

        "model": XGBRegressor(
            random_state=10
        ),

        "params": {
            "model__n_estimators": [100, 200],
            "model__max_depth": [3, 6],
            "model__learning_rate": [0.05, 0.1]
        }
    }
}

## **GRIDSEARCH + PIPLINE**

In [56]:
results=[]

best={}

for name,config in models.items():
    print(name)

    p = Pipeline(
        steps=[
            ('preprocessing',preprocessor),
            ('model',config['model'])
        ]
    )

    grid=GridSearchCV(
        estimator=p,
        param_grid=config['params'],
        cv=5,
        scoring={
        "r2": "r2",
        "mae": "neg_mean_absolute_error"},
        refit='r2',
        n_jobs=-1
    )

    grid.fit(x_train,y_train)

    best[name]= grid.best_estimator_

    y_pred = grid.best_estimator_.predict(x_test)

    mae= mean_absolute_error(y_test,y_pred)
    rmse=mean_squared_error(y_test,y_pred)**0.5
    r2=r2_score(y_test,y_pred)

    results.append({

        "Model": name,

        "MAE": mae,

        "RMSE": rmse,

        "R2": r2

    })

Linear Regression
Random Forest


/home/yassine/Projets/Driver-station/.venv/lib/python3.14/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/home/yassine/Projets/Driver-station/.venv/lib/python3.14/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/home/yassine/Projets/Driver-station/.venv/lib/python3.14/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/home/yassine/Projets/Driver-station/.venv/lib/python3.14/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [0] during transform. Thes

SVR


/home/yassine/Projets/Driver-station/.venv/lib/python3.14/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/home/yassine/Projets/Driver-station/.venv/lib/python3.14/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/home/yassine/Projets/Driver-station/.venv/lib/python3.14/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/home/yassine/Projets/Driver-station/.venv/lib/python3.14/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [0] during transform. Thes

XGBoost


/home/yassine/Projets/Driver-station/.venv/lib/python3.14/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/home/yassine/Projets/Driver-station/.venv/lib/python3.14/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/home/yassine/Projets/Driver-station/.venv/lib/python3.14/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/home/yassine/Projets/Driver-station/.venv/lib/python3.14/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [0] during transform. Thes

## **RESULTAT** 

In [57]:
r=pd.DataFrame(results)

print(r)

               Model            MAE           RMSE        R2
0  Linear Regression  215599.767968  439439.005529  0.454511
1      Random Forest  199327.793127  435709.473029  0.463731
2                SVR  244315.254631  570602.948832  0.080278
3            XGBoost  190933.984375  405816.344353  0.534792


## **CHOISIR LE MODEL** 

In [58]:
joblib.dump(
    best["XGBoost"],
    "../model/model.pkl"
)

['../model/model.pkl']